# 05 — Train a Layerwise DQN Model Online

Train a MOUSE model with **layerwise DQN**: every backbone layer gets its own Q head, and `LayerwiseDqnObjective` applies a one-step Bellman loss at each depth.

Discounts are anchored at the endpoints: layer `0` uses each `gamma_*_start`, the deepest layer uses the deep value (`gamma_step`, `gamma_episode_terminal`, …). Intermediate layers get **linearly increasing effective horizon**:

`H(gamma) = 1 / (1 - gamma)`

`H_l = H_start + (H_deep - H_start) * (l / (L - 1))`, then `gamma_l = 1 - 1 / H_l`.

For held-out evaluation after training, use `examples/04_inference.ipynb`. This example truncates the backbone with `num_layers=4`.


In [1]:
from collections import deque

import torch

from mouse_envs import EnvConfig, make_env
from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import LayerwiseDiscreteActionValueHead
from mouse_core.objectives import LayerwiseDqnObjective


MODEL_ID = "mouse-example-model"  # Hub repo id for push_model_to_hub
MAX_ACTIONS = 4  # discrete action head size (FrozenLake: up/down/left/right)
MAX_OBS_DISCRETE = 64  # observation embedder vocab (max cells in 8×8 maps)
NUM_ENVS = 1000  # parallel SingleEnv instances (one Datastore each)

GRADIENT_STEPS = 20000  # total optimizer updates (same budget as 02_train_offline)
SEQUENCE_LENGTH = 512  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4  # sequences per optimizer step

ENV_STEPS_PER_CYCLE = 1000  # env transitions collected each cycle
STEPS_PER_ENV = 100  # consecutive env steps on one env before rotating
GRADIENT_STEPS_PER_CYCLE = 1000  # optimizer updates each cycle (once learning starts)

LEARNING_STARTS = 2000  # env transitions before the first optimizer update
EXPLORATION_ENDS = 10000  # env step when linear ε decay from 1.0 to 0.0 finishes


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Online training keeps the environment in the training loop. Each `SingleEnv` instance runs one infinite task with deterministic FrozenLake dynamics and 50-step episode truncation, so the model keeps carrying policy context across episode boundaries.


In [2]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "is_slippery": False,
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            #"step_penalty": -0.01,
            "max_episode_steps": 50,
            "map_seed": i,
        },
    )
    for i in range(NUM_ENVS)
]

envs = [make_env(config) for config in configs]
print(f"Environments {min(10, len(envs))} of {len(envs)}:")
for env in envs[:10]:
    print(env.name)

Environments 10 of 1000:
proc_frozenlake_online_0
proc_frozenlake_online_1
proc_frozenlake_online_2
proc_frozenlake_online_3
proc_frozenlake_online_4
proc_frozenlake_online_5
proc_frozenlake_online_6
proc_frozenlake_online_7
proc_frozenlake_online_8
proc_frozenlake_online_9


## Build the model

`LayerwiseDiscreteActionValueHead` needs one Q head per transformer block. Set `num_backbone_layers=len(backbone.model.layers)` and pass the same count to `LayerwiseDqnObjective`.

Pass `num_layers=4` to truncate the backbone for a quicker demo run.


In [3]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")
num_backbone_layers = len(backbone.model.layers)

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

head = LayerwiseDiscreteActionValueHead(
    num_backbone_layers=num_backbone_layers,
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
print(f"Backbone layers: {num_backbone_layers}")
print(model)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Backbone layers: 28
Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(64, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2

## Replay buffer and policy contexts

Each env writes to one `Datastore`; together they are the live replay buffer. Each env also has a `contexts` deque capped at `SEQUENCE_LENGTH` — the action-selection history used during collection. Contexts are never cleared; episode boundaries appear as `done` values in the sequence.


In [4]:
stores = [Datastore(name=env.name) for env in envs]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in envs]


## Rollout phase

One **rollout** gathers up to `ENV_STEPS_PER_CYCLE` env transitions, visiting envs in order and taking up to `STEPS_PER_ENV` consecutive transitions per env before moving on. For each env visit the loop uses a single **B=1 KV cache** (like vLLM per-sequence caching): full-context pass on the first model call, then incremental passes for new rows only. The cache is discarded when moving to the next env — inference is never batched across envs, so only one cache is needed at a time.

1. Choose actions with epsilon-greedy exploration (`epsilon` is recomputed from the global env-step counter on every transition). Random actions skip the model; the next model call feeds all uncached rows as a catch-up slice.
2. Step the env, append to its `Datastore` and `contexts` deque. The KV cache grows incrementally for the duration of that env's visit and is discarded when the loop moves to the next env.

Each rollout clears every env tracker first, then prints mean reward/length for episodes completed during the burst when the rollout finishes.


In [5]:
def epsilon_for_env_step(env_step: int) -> float:
    """Linear ε decay: 1.0 (explore) → 0.0 (greedy)."""
    if EXPLORATION_ENDS <= 0:
        raise ValueError("EXPLORATION_ENDS must be positive.")
    frac = min(env_step / EXPLORATION_ENDS, 1.0)
    return 1.0 - frac


In [6]:
def rollout(
    model: Model,
    envs: list,
    stores: list[Datastore],
    contexts: list[deque],
    env_steps: int,
    grad_steps: int,
) -> int:
    """Collect up to ``ENV_STEPS_PER_CYCLE`` env transitions, then return."""
    for env in envs:
        env.tracker.clear()
    model.eval()

    budget = ENV_STEPS_PER_CYCLE
    collected = 0

    for env, store, context in zip(envs, stores, contexts):
        if collected >= budget:
            break

        kv_cache = None
        cache_count = 0
        ctx_count = 0
        chunk = min(STEPS_PER_ENV, budget - collected)

        for _ in range(chunk):
            epsilon = epsilon_for_env_step(env_steps)
            if not context or torch.rand(1).item() < epsilon:
                inp = env.sample_random_input()
            else:
                ctx_list = list(context)
                with torch.no_grad():
                    if kv_cache is None:
                        preds, _, kv_cache = model([ctx_list], use_cache=True)
                    else:
                        uncached = ctx_count - cache_count
                        preds, _, kv_cache = model(
                            [ctx_list[-uncached:]], cache=kv_cache, use_cache=True
                        )
                    cache_count = ctx_count

                action = model.get_action(preds, temperature=0.0, num_actions=MAX_ACTIONS)
                inp = {"action": action.squeeze().cpu()}

            out = env.step(inp)
            store.append({**inp, **out})
            context.append({**inp, **out})
            ctx_count += 1
            env_steps += 1
            collected += 1

    rewards: list[float] = []
    lengths: list[int] = []
    for env in envs:
        rewards.extend(env.tracker.episode_cum_rewards)
        lengths.extend(env.tracker.episode_lengths)
    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).float().mean().item() if n else float("nan")
    epsilon = epsilon_for_env_step(env_steps)
    print(
        f"env_step={env_steps} grad_step={grad_steps} epsilon={epsilon:.3f}"
        f" | {n} completed episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}"
    )
    return env_steps


## Training phase

One **train burst** runs after each rollout once `env_steps >= LEARNING_STARTS` and prints compact per-layer stats (`L{i}:loss/q`) plus deepest-layer `q*`.

With `gamma_episode_terminal_start=0.0` and `gamma_episode_terminal=0.99`, horizons go from `1` to `100` steps evenly across four layers.


In [7]:
loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    weight_mode="per_step",
    pack=True,
    num_workers=0,
)
objective = LayerwiseDqnObjective(
    num_backbone_layers=num_backbone_layers,
    gamma_step_start=1.0,
    gamma_step=1.0,
    gamma_episode_terminal_start=0.0,
    gamma_episode_terminal=0.99,
    gamma_episode_truncated_start=0.0,
    gamma_episode_truncated=0.99,
    gamma_task_terminal_start=0.0,
    gamma_task_terminal=0.99,
    gamma_task_truncated_start=0.0,
    gamma_task_truncated=0.99,
    tau=0.0005,
)
print(f"layer_gamma_step={objective.layer_gamma_step}")
print(f"layer_gamma_episode_terminal={objective.layer_gamma_episode_terminal}")
print(
    "horizons=",
    [round(LayerwiseDqnObjective.effective_horizon(g), 2) for g in objective.layer_gamma_episode_terminal],
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)


layer_gamma_step=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
layer_gamma_episode_terminal=[0.0, 0.7857142857142856, 0.8799999999999999, 0.9166666666666666, 0.9361702127659574, 0.9482758620689655, 0.9565217391304347, 0.9624999999999999, 0.967032967032967, 0.9705882352941176, 0.9734513274336283, 0.9758064516129032, 0.9777777777777777, 0.9794520547945206, 0.9808917197452229, 0.9821428571428571, 0.9832402234636871, 0.9842105263157894, 0.9850746268656716, 0.9858490566037735, 0.9865470852017937, 0.9871794871794871, 0.9877551020408163, 0.98828125, 0.9887640449438202, 0.9892086330935251, 0.9896193771626297, 0.99]
horizons= [1.0, 4.67, 8.33, 12.0, 15.67, 19.33, 23.0, 26.67, 30.33, 34.0, 37.67, 41.33, 45.0, 48.67, 52.33, 56.0, 59.67, 63.33, 67.0, 70.67, 74.33, 78.0, 81.67, 85.33, 89.0, 92.67, 96.33, 100.0]


In [8]:
def train_burst(
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: LayerwiseDqnObjective,
    loader: DataLoader,
    env_steps: int,
    grad_steps: int,
    burst_steps: int,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh replay and run ``burst_steps`` optimizer updates."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(burst_steps):
        batch = loader.next_batch()

        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.polyak_update(action_value_layerwise_tau=objective.tau)

    assert loss is not None
    layer_stats = " ".join(
        f"L{i}:{metrics[f'layer_{i}_loss']:.3f}/{metrics[f'layer_{i}_q_mean']:.2f}"
        for i in range(objective.num_backbone_layers)
    )
    print(
        f"env_step={env_steps} grad_step={grad_steps + burst_steps}"
        f" | loss={metrics['action_value_layerwise']:.4f} q*={metrics['q_values_mean']:.2f}"
        f" | {layer_stats}"
    )
    return loss, metrics


## Run online training

The master loop alternates **rollout → train** until `GRADIENT_STEPS` optimizer updates have been applied — the same total as offline training. Optimizer updates are skipped until `LEARNING_STARTS` env transitions have been collected. Each burst prints stats at the end of `rollout` / `train_burst`.


In [ ]:
env_steps = 0
grad_steps = 0

while grad_steps < GRADIENT_STEPS:
    env_steps = rollout(model, envs, stores, contexts, env_steps, grad_steps)

    if env_steps >= LEARNING_STARTS:
        burst = min(GRADIENT_STEPS_PER_CYCLE, GRADIENT_STEPS - grad_steps)
        train_burst(model, optimizer, objective, loader, env_steps, grad_steps, burst)
        grad_steps += burst

loader.close()

for env in envs:
    env.close()

print(f"Online layerwise DQN finished ({grad_steps} optimizer steps, {env_steps} env steps).")


env_step=1000 grad_step=0 epsilon=0.900 | 107 completed episodes | mean reward 0.056 | mean length 7.6
env_step=2000 grad_step=0 epsilon=0.800 | 115 completed episodes | mean reward 0.026 | mean length 7.5


## Push to the Hub

Run this cell if you want to save the online-trained checkpoint to the shared Hub repo under `MODEL_ID`. The inference notebook loads the current checkpoint from that repo, so whichever training notebook you push last is the model it evaluates.

In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")